# 03. Match posts to the 38 topics

Scores every post against each of the 38 topic sentences with BERTScore recall. Recall
rather than precision or F1 because the question is whether a post *mentions* a topic,
not whether the post is about nothing else: a post can touch several topics at once,
and each gets its own score.

These scores are `r_ij` in Eq. (2), and they become the country topic weights `PP`
in notebook 05.

## Same settings, different plumbing

Model, layer, and options are exactly the 2025 run's: `microsoft/deberta-xlarge-mnli`,
`idf=False`, `rescale_with_baseline=True`, `lang='en'`, negative recalls clipped to
zero. Posts are scored on `Twitter_processed`, the cleaned token string, not the raw
text — the same input the original used.

What changes is that the 38 topic sentences are embedded once instead of once per
post. The 2025 code called `bertscore.compute(predictions=[post]*38, references=topics)`
in a loop, which re-encodes the topic sentences for every post; at ~16 posts/s that
was around ten hours. Section 2 checks the batched path reproduces the published
numbers before anything is run at scale.

## 0. Setup

Runs in two places. Locally it reads and writes the pipeline's `output/` folder. On
Colab it mounts Drive and works in a folder there, because the GPU is the reason to be
on Colab at all: an M1 Pro manages 12.5 posts/s on this model, an A100 well over a
hundred.

Only two things need to reach Colab: `posts_processed.parquet` (38 MB, produced by
`posts_preprocess.py`) and `assets/topic_sentences.json`. The 170 GB of raw posts stays
on the laptop — it has already done its job in notebook 02.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))  # the modules the notebooks import live in src/


In [ ]:
!pip install -q bert_score transformers torch

In [ ]:
import os
import sys
import time

import numpy as np
import pandas as pd

ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/Climate change project/revision'
    sys.path.insert(0, WORK)          # config.py, bertscore_fast.py, assets/ go here too
else:
    WORK = os.path.dirname(os.path.abspath('.'))
    sys.path.insert(0, os.getcwd())

import config as cfg
import bertscore_fast as bf

IN_DIR = WORK if ON_COLAB else cfg.OUTPUT_DIR
OUT_DIR = os.path.join(IN_DIR, 'topic_scores')
os.makedirs(OUT_DIR, exist_ok=True)

device = bf.pick_device()             # cuda, then Apple mps, then cpu
print("colab  :", ON_COLAB)
print("device :", device)
print("in     :", IN_DIR)
print("topic sentences:", len(cfg.TOPIC_SENTENCES))

## 1. Load the preprocessed posts

Cleaning happens in `posts_preprocess.py`, run locally before uploading: lowercase,
letters only, stop words out, lemmatise, drop words occurring fewer than twice, join
bigrams. It takes about 40 seconds on a laptop and there is no reason to spend GPU
time on it.

Two differences from 2025 are worth keeping in mind when comparing scores. The input
is `full_text` rather than the truncated `text`; and the rare-word cut and the bigram
model are fitted on the corpus, which is now 590,488 posts rather than 338,520, so the
vocabulary differs slightly. Scores for a given post will not match the published file
exactly even where nothing else changed — section 2 checks the *method* rather than
the numbers, by rescoring posts from the old sample with the old vocabulary.

In [ ]:
posts = pd.read_parquet(os.path.join(IN_DIR, 'posts_processed.parquet'))
print(f"posts: {len(posts):,}")
print(f"empty after cleaning: {(posts.Twitter_processed.str.len() == 0).sum()}")
posts.head(3)

## 2. Check the scoring against the published run

Before scoring 590,488 posts, rescore posts the 2025 run already scored and compare.
`assets/validation_sample.csv` holds 200 of them together with the recall values that
run recorded — small enough to travel with the code, so the check works on Colab as
well as on a laptop.

Worth running wherever the scoring happens, not just once: CUDA and Apple's MPS are
not obliged to produce bit-identical results, and this is the cheapest way to find out
that they do. Measured on an M1 Pro: maximum absolute difference 1.1e-06.

In [ ]:
scorer = bf.TopicScorer(cfg.TOPIC_SENTENCES, device=device, batch_size=16)
print(f"layer {scorer.num_layers}, baseline {scorer.baseline:.7f}")

report = bf.validate_against(scorer=scorer)
for k, v in report.items():
    print(f"  {k:<20s} {v}")

assert report['max_abs_diff'] < 1e-3, "batched scores disagree with the published run"
print("\nAgreement confirmed.")

In [ ]:
### How fast is it here?

sample = posts.Twitter_processed.head(64).tolist()
t0 = time.time()
_ = scorer.score(sample)
rate = len(sample) / (time.time() - t0)
print(f"{rate:.1f} posts/s on {device}")
print(f"projected for {len(posts):,} posts: {len(posts)/rate/3600:.1f} h")
print("\nThe 2025 loop managed about 16 posts/s on a Colab GPU.")

## 3. Score the corpus

Sharded so a dropped Colab session resumes rather than restarts. Each shard is a
parquet of `tweet_id` plus `Topic_1..Topic_38`, written straight to Drive.

In [ ]:
SHARD = 20000
RUN_FULL = False        # set True once section 2 passes and the rate looks acceptable

if RUN_FULL:
    texts = posts.Twitter_processed.tolist()
    ids = posts.tweet_id.tolist()
    cols = cfg.tw_columns()
    t0 = time.time()
    for s in range(0, len(texts), SHARD):
        path = os.path.join(OUT_DIR, f'shard_{s//SHARD:04d}.parquet')
        if os.path.exists(path):
            continue
        scores = scorer.score(texts[s:s + SHARD])
        out = pd.DataFrame(scores, columns=cols)
        out.insert(0, 'tweet_id', ids[s:s + SHARD])
        out.to_parquet(path, index=False)
        done = min(s + SHARD, len(texts))
        el = time.time() - t0
        print(f"[{done:>7,}/{len(texts):,}] {el/60:.0f} min, "
              f"ETA {(len(texts)-done)*el/max(done,1)/60:.0f} min", flush=True)
else:
    print("RUN_FULL is False.")

## 4. Assemble

One row per post, 38 columns of recall. This is what notebook 05 aggregates into the
country topic weights.

In [ ]:
import glob

shards = sorted(glob.glob(os.path.join(OUT_DIR, 'shard_*.parquet')))
print("shards:", len(shards))

scores = pd.concat([pd.read_parquet(f) for f in shards], ignore_index=True)
scores.to_parquet(os.path.join(IN_DIR, 'topic_recall_2022.parquet'), index=False)

cols = cfg.tw_columns()
print(f"rows {len(scores):,}   expected {len(posts):,}")
print(f"all scores in [0, 1]: {bool((scores[cols] >= 0).all().all() and (scores[cols] <= 1).all().all())}")
all_zero = int((scores[cols].sum(axis=1) == 0).sum())
print(f"posts matching no topic at all: {all_zero:,} ({100*all_zero/len(scores):.1f}%)")

print("\nMean recall per topic, highest first:")
mean = scores[cols].mean().sort_values(ascending=False)
# topic_table.csv comes from notebook 01 and is not part of the Colab bundle
tbl_path = os.path.join(IN_DIR, 'topic_table.csv')
tbl = pd.read_csv(tbl_path).set_index('topic') if os.path.exists(tbl_path) else None
for c in mean.head(10).index:
    t = c.replace('Topic_', 'T')
    label = tbl.loc[t, 'sentence'].split(' - ')[0][4:] if tbl is not None else \
        cfg.TOPIC_SENTENCES[int(c.split('_')[1]) - 1].split(' - ')[0]
    print(f"  {t:<5s} {mean[c]:.4f}  {label}")

## 5. Does the filter noise matter?

Precision of the taxonomy is 67% overall and 50% for the posts that only satisfy a
conjunction rule (notebook 02, section 7). The worry is that those posts distort the
country topic profiles.

They should not, much: BERTScore is a soft match, so a post about a job opening scores
near zero against every topic and contributes almost nothing once the country weights
are normalised. This section checks that rather than assuming it.

In [ ]:
strong = pd.read_parquet(os.path.join(cfg.OUTPUT_DIR, 'strong_flag.parquet')).strong.values
tot = scores[cols].sum(axis=1)

print("Total recall across the 38 topics, by rule type")
print(f"  names climate directly : {tot[strong].mean():.4f}")
print(f"  conjunction rules only : {tot[~strong].mean():.4f}")
print(f"  ratio                  : {tot[~strong].mean()/tot[strong].mean():.2f}")
print(f"\nShare of all topic weight carried by conjunction-only posts: "
      f"{100*tot[~strong].sum()/tot.sum():.1f}%")
print(f"Their share of posts:                                        "
      f"{100*(~strong).mean():.1f}%")
print("\nIf the first number is well below the second, the noisier posts are already "
      "down-weighted by the matching itself, and the sensitivity check in notebook 08 "
      "should confirm the indices barely move.")